## Project 4 — Deepfake Generation & Detection 🍎 ↔️ 🍊

Welcome to Project 4! In this notebook, we're going to dive into the fascinating world of Diffusion Models — but this time, instead of just generating random images from scratch, we're going to build a targeted pipeline that can actually swap objects inside an image, turning apples into oranges (and vice versa)!

You'll build everything step-by-step, from preparing the dataset to building the neural networks, training them, and finally creating your own fruit *deepfakes*. Let's get started! 🚀

### Outline

1. **Grab the data** — pull a preprocessed fruit dataset from the repo and peek at some raw samples.
2. **The VAE** — load a frozen Stable Diffusion VAE that translates between pixels and a compact latent space.
3. **Pre-encode the dataset** — compress every image into 4×16×16 latents, once.
4. **Wrap the latents in a DataLoader** — standard PyTorch plumbing, just in latent space.
5. **Build the model** — a Fruit Encoder, a conditioned U-Net with cross-attention, and a DDPM noise schedule.
6. **Train it** — train the diffusion model.
7. **See what it learned** — reconstructions and apple↔orange deepfakes.
8. **Catching the fakes** — generate a batch of deepfakes, download them, and switch to the discriminator notebook to train a detector.

### Step 1 — Grab the data 🧺

We start by pulling the preprocessed fruit dataset from the repo (via git LFS), installing dependencies, and peeking at a handful of raw training samples before anything else touches them. This is the only place you'll see pixel-space images before we switch to latents.

In [ ]:
#@title Fetch dataset (this might take around 5-7 minutes) { display-mode: "form" }
# === Clone repo and pull LFS files ===
!git lfs install
!git clone https://github.com/eth-bmai-fs26/project.git
%cd project
!git checkout week4/deepfake
!git lfs pull
%cd /content

DATASET_DIR = 'project/week4/deepfake/dataset'

# Verify files downloaded correctly
import os
for f in ['fruits_train.pt', 'fruits_test.pt']:
    size = os.path.getsize(f'{DATASET_DIR}/{f}') / (1024**2)
    print(f"{f}: {size:.0f} MB")

In [ ]:
#@title Install dependencies and imports { display-mode: "form" }

!pip install -q diffusers accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from diffusers import AutoencoderKL
import numpy as np
import matplotlib.pyplot as plt
import random
import os

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
#@title Preview training samples { display-mode: "form" }
# Preview a few training samples in pixel space before we encode them.
# This is the only place you'll see raw pixel images before training —
# everything after this step happens in VAE latent space.
import torch
import matplotlib.pyplot as plt

preview = torch.load(f'{DATASET_DIR}/fruits_train.pt', weights_only=False)
CLASS_NAMES = {0: 'apple', 1: 'orange'}

n = 5
idxs = torch.randperm(len(preview['labels']))[:n]

fig, axes = plt.subplots(4, n, figsize=(3 * n, 10))
row_labels = ['Composite', 'Mask', 'Crop (fruit)', 'Redacted (background)']

for col, idx in enumerate(idxs):
    lbl = preview['labels'][idx].item()
    axes[0, col].imshow(preview['composites'][idx].float().permute(1, 2, 0).numpy())
    axes[0, col].set_title(CLASS_NAMES[lbl], fontsize=10)
    axes[1, col].imshow(preview['masks'][idx, 0].float().numpy(), cmap='gray')
    axes[2, col].imshow(preview['crops'][idx].float().permute(1, 2, 0).numpy())
    axes[3, col].imshow(preview['redacted'][idx].float().permute(1, 2, 0).numpy())
    for row in range(4):
        axes[row, col].axis('off')

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, labelpad=80, va='center')

plt.suptitle('Training data — pixel space (before VAE encoding)', fontsize=14)
plt.tight_layout()
plt.show()

del preview


### Step 2 — The VAE, our pixel ↔ latent translator 🔁

Instead of training on raw pixels, we work in the compact latent space of a pretrained Stable Diffusion VAE. We load it, freeze it (we never fine-tune the VAE), and do a quick encode/decode sanity check so you can see that a 128×128 image survives the round-trip through a 4×16×16 latent.

In [ ]:
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float32
).to(device)

# Freeze completely — we never train the VAE
vae.eval()
for param in vae.parameters():
    param.requires_grad = False

In [ ]:
#@title Test the VAE { display-mode: "form" }
# Test the VAE: encode then decode an image
dummy = torch.randn(1, 3, 128, 128).to(device)
with torch.no_grad():
    latent = vae.encode(dummy).latent_dist.sample()
    reconstructed = vae.decode(latent).sample

print(f"Input:   {dummy.shape}")           # (1, 3, 128, 128)
print(f"Latent:  {latent.shape}")          # (1, 4, 16, 16)
print(f"Decoded: {reconstructed.shape}")   # (1, 3, 128, 128)
print(f"Compression: {128*128*3} pixels → {16*16*4} latents ({128*128*3 / (16*16*4):.0f}× reduction)")
print("VAE loaded ✅")

### Step 3 — Pre-encode the dataset 🗜️

Running the VAE during every training step would be wasteful since we never update it. Instead we encode every image **once** and save the resulting 4×16×16 latents to disk. After this step, the VAE doesn't run again until we want to visualize results.

In [ ]:
#@title Pre-encode dataset to latent space { display-mode: "form" }
# This is done ONCE and saved — no need to run the VAE during training

def encode_dataset_to_latents(data_path, vae, device, batch_size=16):
    data = torch.load(data_path, weights_only=False)

    # Convert float16 to float32 for VAE
    composites = data['composites'].float()
    masks = data['masks'].float()
    redacted = data['redacted'].float()
    crops = data['crops'].float()
    labels = data['labels']

    print(f"Encoding {len(labels)} images to latent space...")

    def to_vae_range(x):
        return x * 2.0 - 1.0

    latent_composites = []
    latent_redacted = []
    latent_crops = []
    downsampled_masks = []

    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch_comp = to_vae_range(composites[i:i+batch_size]).to(device)
            batch_red = to_vae_range(redacted[i:i+batch_size]).to(device)
            batch_crop = to_vae_range(crops[i:i+batch_size]).to(device)
            batch_mask = masks[i:i+batch_size]

            z_comp = vae.encode(batch_comp).latent_dist.sample() * 0.18215
            z_red = vae.encode(batch_red).latent_dist.sample() * 0.18215
            z_crop = vae.encode(batch_crop).latent_dist.sample() * 0.18215

            mask_down = F.interpolate(batch_mask, size=(16, 16), mode='nearest')

            latent_composites.append(z_comp.cpu())
            latent_redacted.append(z_red.cpu())
            latent_crops.append(z_crop.cpu())
            downsampled_masks.append(mask_down.cpu())

            if (i // batch_size) % 20 == 0:
                print(f"  {i}/{len(labels)}")

    # Free the pixel-space data
    del composites, masks, redacted, crops
    import gc; gc.collect()

    latent_data = {
        'composites': torch.cat(latent_composites),
        'redacted': torch.cat(latent_redacted),
        'crops': torch.cat(latent_crops),
        'masks': torch.cat(downsampled_masks),
        'labels': labels,
    }

    print(f"Composites latent shape: {latent_data['composites'].shape}")
    print("Done ✅")
    return latent_data

latent_train = encode_dataset_to_latents(f'{DATASET_DIR}/fruits_train.pt', vae, device)
latent_test = encode_dataset_to_latents(f'{DATASET_DIR}/fruits_test.pt', vae, device)

# Save locally (no Drive needed)
torch.save(latent_train, 'latent_train.pt')
torch.save(latent_test, 'latent_test.pt')
del latent_train, latent_test
import gc; gc.collect()
print("Latent datasets saved ✅")

### Step 4 — Wrap the latents in a DataLoader 🚀

The pre-encoded tensors live on disk as 4×16×16 VAE latents. Here we wrap them in a small `Dataset` so PyTorch can batch them for training. Nothing from this step onward touches pixels — everything happens in latent space.

In [ ]:
BATCH_SIZE = 512

class LatentFruitDataset(Dataset):
    def __init__(self, data_path):
        data = torch.load(data_path, weights_only=False)
        self.composites = data['composites']  # (N, 4, 16, 16)
        self.masks = data['masks']             # (N, 1, 16, 16)
        self.redacted = data['redacted']       # (N, 4, 16, 16)
        self.crops = data['crops']             # (N, 4, 16, 16)
        self.labels = data['labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'composite': self.composites[idx],
            'mask': self.masks[idx],
            'redacted': self.redacted[idx],
            'crop': self.crops[idx],
            'label': self.labels[idx],
        }

train_dataset = LatentFruitDataset('latent_train.pt')
test_dataset = LatentFruitDataset('latent_test.pt')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

In [ ]:
#@title Verify Latent DataLoader { display-mode: "form" }
batch = next(iter(train_loader))
print(f"Latent composite: {batch['composite'].shape}")  # (64, 4, 16, 16)
print(f"Latent mask:      {batch['mask'].shape}")        # (64, 1, 16, 16)
print(f"Latent redacted:  {batch['redacted'].shape}")    # (64, 4, 16, 16)
print(f"Latent crop:      {batch['crop'].shape}")        # (64, 4, 16, 16)
print("Latent DataLoader ready ✅")

### Step 5 — Build the model 🏗️

Three pieces come together: a **FruitEncoder** that summarizes a fruit crop into a conditioning vector, a **LatentUNet** that predicts noise using spatial conditioning (redacted background + mask) plus semantic conditioning through cross-attention, and a **DDPMScheduler** that manages the forward/reverse noise process.

In [ ]:
# Now operates on 4×16×16 latents instead of 3×128×128 pixels

class LatentFruitEncoder(nn.Module):
    """
    Encode a fruit's latent representation into a conditioning embedding.
    Input: VAE latent (4, 16, 16)
    Output: embedding vector (128,)
    """
    def __init__(self, in_channels=4, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, 2, 1),  nn.GroupNorm(8, 64),  nn.GELU(),
            nn.Conv2d(64, 128, 3, 2, 1),           nn.GroupNorm(8, 128), nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, embed_dim),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = torch.log(torch.tensor(10000.0, device=t.device)) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class CrossAttention(nn.Module):
    def __init__(self, feature_dim, embed_dim=128, num_heads=4, num_tokens=4):
        super().__init__()
        self.num_heads = num_heads
        self.num_tokens = num_tokens
        self.head_dim = feature_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.to_kv_tokens = nn.Sequential(
            nn.Linear(embed_dim, feature_dim * num_tokens),
            nn.GELU(),
            nn.Linear(feature_dim * num_tokens, feature_dim * num_tokens * 2),
        )
        self.to_q = nn.Linear(feature_dim, feature_dim)
        self.to_out = nn.Sequential(nn.Linear(feature_dim, feature_dim), nn.GELU())
        self.norm_features = nn.LayerNorm(feature_dim)
        self.norm_out = nn.LayerNorm(feature_dim)

    def forward(self, x, cond):
        B, C, H, W = x.shape
        residual = x

        x_flat = x.permute(0, 2, 3, 1).reshape(B, H * W, C)
        x_flat = self.norm_features(x_flat)
        q = self.to_q(x_flat)

        kv = self.to_kv_tokens(cond)
        kv = kv.reshape(B, self.num_tokens, 2, C)
        k, v = kv[:, :, 0], kv[:, :, 1]

        q = q.reshape(B, H * W, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = k.reshape(B, self.num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.reshape(B, self.num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = torch.matmul(attn, v)

        out = out.permute(0, 2, 1, 3).reshape(B, H * W, C)
        out = self.to_out(out)
        out = self.norm_out(out)
        out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)

        return residual + out


class LatentUNet(nn.Module):
    """
    U-Net operating in VAE latent space.
    Input: 16×16 latents. Only 2 downsampling levels: 16→8→4

    Spatial input: noisy_latent (4ch) + redacted_latent (4ch) + mask (1ch) = 9 channels
    Output: predicted noise (4 channels, matching latent dim)
    """
    def __init__(self, in_channels=9, out_channels=4,
                 base_ch=128, embed_dim=128):
        super().__init__()
        ch = base_ch

        # Time + condition embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(ch),
            nn.Linear(ch, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

        # Encoder: 16→8→4
        self.enc1 = ConvBlock(in_channels, ch)        # 128ch, 16×16
        self.enc2 = ConvBlock(ch, ch * 2)             # 256ch, 8×8
        self.down = nn.MaxPool2d(2)

        # Bottleneck: 4×4
        self.bottleneck = ConvBlock(ch * 2, ch * 2)   # 256ch, 4×4

        # Decoder: 4→8→16
        self.up2 = nn.ConvTranspose2d(ch * 2, ch * 2, 2, 2)
        self.dec2 = ConvBlock(ch * 4, ch)             # skip: 256+256=512 → 128

        self.up1 = nn.ConvTranspose2d(ch, ch, 2, 2)
        self.dec1 = ConvBlock(ch * 2, ch)             # skip: 128+128=256 → 128

        self.out_conv = nn.Conv2d(ch, out_channels, 1)

        # Cross-attention at enc2 (8×8) and bottleneck (4×4)
        self.attn_enc1 = CrossAttention(feature_dim=ch, embed_dim=embed_dim)
        self.attn_enc2 = CrossAttention(feature_dim=ch * 2, embed_dim=embed_dim)
        self.attn_bottleneck = CrossAttention(feature_dim=ch * 2, embed_dim=embed_dim)
        self.attn_dec2 = CrossAttention(feature_dim=ch, embed_dim=embed_dim)
        self.attn_dec1 = CrossAttention(feature_dim=ch, embed_dim=embed_dim)

    def forward(self, x_noisy, t, redacted, mask, fruit_embed):
        # Combine time and fruit conditioning
        t_emb = self.time_mlp(t)
        cond = t_emb + fruit_embed

        # Spatial input
        x = torch.cat([x_noisy, redacted, mask], dim=1)  # (B, 9, 16, 16)

        # Encoder
        e1 = self.enc1(x)
        e1 = self.attn_enc1(e1, cond)
        e1 = self.enc1(x)                              # (B, 128, 16, 16)
        e2 = self.enc2(self.down(e1))                   # (B, 256, 8, 8)
        e2 = self.attn_enc2(e2, cond)

        # Bottleneck
        b = self.bottleneck(self.down(e2))              # (B, 256, 4, 4)
        b = self.attn_bottleneck(b, cond)

        # Decoder
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))  # (B, 128, 8, 8)
        d2 = self.attn_dec2(d2, cond)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # (B, 128, 16, 16)
        d1 = self.attn_dec1(d1, cond)

        return self.out_conv(d1)  # (B, 4, 16, 16)

In [ ]:
class DDPMScheduler:
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.num_timesteps = num_timesteps
        self.device = device

        betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

    def add_noise(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        sqrt_alpha = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_alpha * x_0 + sqrt_one_minus * noise, noise

    @torch.no_grad()
    def sample(self, model, fruit_encoder, crop, redacted, mask,
               return_intermediates=False, intermediate_steps=10):
        shape = (crop.shape[0], 4, 16, 16)  # latent shape
        x = torch.randn(shape, device=self.device)
        fruit_embed = fruit_encoder(crop)

        intermediates = []
        save_every = max(1, self.num_timesteps // intermediate_steps)

        for t_val in reversed(range(self.num_timesteps)):
            B = x.shape[0]
            t = torch.full((B,), t_val, device=self.device, dtype=torch.long)

            pred_noise = model(x, t, redacted, mask, fruit_embed)

            alpha_t = self.alphas[t_val]
            alpha_bar_t = self.alphas_cumprod[t_val]
            beta_t = self.betas[t_val]

            coeff = (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)
            mean = (1 / torch.sqrt(alpha_t)) * (x - coeff * pred_noise)

            if t_val > 0:
                x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
            else:
                x = mean

            if return_intermediates and (t_val % save_every == 0 or t_val == 0):
                intermediates.append(x.cpu().clone())

        if return_intermediates:
            return x, intermediates
        return x

### Step 6 — Train it! 🏋️‍♀️

Joint training of the U-Net and FruitEncoder on the standard denoising objective. We use Adam with square-root LR scaling relative to batch size, a short linear warmup, cosine decay, and gradient clipping to keep things stable.

In [ ]:
LATENT_IMG_SIZE = 16
NUM_EPOCHS = 300
EMBED_DIM = 128
NUM_TIMESTEPS = 1000
BASE_BATCH_SIZE = 64
BASE_LR = 5e-5
LEARNING_RATE = BASE_LR * (BATCH_SIZE / BASE_BATCH_SIZE) ** 0.5

CLASS_NAMES = {0: 'apple', 1: 'orange'}

unet = LatentUNet(base_ch=640, embed_dim=EMBED_DIM).to(device)
fruit_encoder = LatentFruitEncoder(embed_dim=EMBED_DIM).to(device)
scheduler = DDPMScheduler(num_timesteps=NUM_TIMESTEPS, device=device)

print(f"U-Net params:    {sum(p.numel() for p in unet.parameters()):,}")
print(f"Encoder params:  {sum(p.numel() for p in fruit_encoder.parameters()):,}")


def train_latent_diffusion(unet, fruit_encoder, scheduler, train_loader,
                           device, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    
    optimizer = optim.Adam(
        list(unet.parameters()) + list(fruit_encoder.parameters()), lr=lr
    )

    # Linear warmup → cosine decay. Protects large-batch runs from early divergence.
    WARMUP_EPOCHS = 5
    warmup = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    cosine = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs - WARMUP_EPOCHS
    )
    scheduler_lr = optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
    )
    history = []

    for epoch in range(1, num_epochs + 1):
        unet.train()
        fruit_encoder.train()
        epoch_loss = 0
        n_batches = 0

        for batch in train_loader:
            z_comp = batch['composite'].to(device)    # (B, 4, 16, 16)
            z_mask = batch['mask'].to(device)          # (B, 1, 16, 16)
            z_redacted = batch['redacted'].to(device)  # (B, 4, 16, 16)
            z_crop = batch['crop'].to(device)          # (B, 4, 16, 16)
            B = z_comp.size(0)

            fruit_embed = fruit_encoder(z_crop)

            t = torch.randint(0, scheduler.num_timesteps, (B,), device=device)
            noise = torch.randn_like(z_comp)
            z_noisy, _ = scheduler.add_noise(z_comp, t, noise)

            pred_noise = unet(z_noisy, t, z_redacted, z_mask, fruit_embed)
            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(unet.parameters()) + list(fruit_encoder.parameters()),
                max_norm=1.0
            )
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        scheduler_lr.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{num_epochs} | Loss: {avg_loss:.4f}")

    return history

print(f"U-Net params: {sum(p.numel() for p in unet.parameters()):,}")

history = train_latent_diffusion(unet, fruit_encoder, scheduler, train_loader, device)

### Step 7 — See what it learned 🔍

Two sanity checks: **reconstructions** (give the model a fruit's own crop as conditioning and verify it can rebuild it) and actual **deepfakes** (condition on the opposite class and watch the U-Net paint it in). All latents are decoded back through the frozen VAE before we plot them.

In [ ]:
#@title Decode latents and show reconstructions { display-mode: "form" }

@torch.no_grad()
def decode_latents(vae, latents):
    """Decode VAE latents back to pixel images."""
    latents = latents / 0.18215  # undo the scaling
    images = vae.decode(latents.to(vae.device)).sample
    images = (images + 1.0) / 2.0  # [-1,1] → [0,1]
    return images.clamp(0, 1)


@torch.no_grad()
def show_reconstructions_latent(unet, fruit_encoder, scheduler, vae,
                                test_loader, device, n=6):
    unet.eval()
    fruit_encoder.eval()

    batch = next(iter(test_loader))
    z_crop = batch['crop'][:n].to(device)
    z_redacted = batch['redacted'][:n].to(device)
    z_mask = batch['mask'][:n].to(device)
    z_comp = batch['composite'][:n].to(device)
    labels = batch['label'][:n]

    # Generate in latent space
    z_result = scheduler.sample(unet, fruit_encoder, z_crop, z_redacted, z_mask)

    # Decode both original and generated to pixel space
    orig_images = decode_latents(vae, z_comp)
    gen_images = decode_latents(vae, z_result)

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    fig.suptitle('Top: Original  |  Bottom: Generated (latent diffusion)', fontsize=13)
    for i in range(n):
        axes[0, i].imshow(orig_images[i].cpu().permute(1, 2, 0).numpy())
        axes[0, i].set_title(CLASS_NAMES[labels[i].item()], fontsize=10)
        axes[0, i].axis('off')
        axes[1, i].imshow(gen_images[i].cpu().permute(1, 2, 0).numpy())
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

show_reconstructions_latent(unet, fruit_encoder, scheduler, vae, test_loader, device)

In [ ]:
#@title Deepfake in latent space { display-mode: "form" }

def find_samples_per_class(loader, n_per_class=3):
    collected = {0: [], 1: []}
    for batch in loader:
        for i in range(len(batch['label'])):
            cls = batch['label'][i].item()
            if cls in collected and len(collected[cls]) < n_per_class:
                collected[cls].append({
                    'composite': batch['composite'][i],
                    'mask': batch['mask'][i],
                    'redacted': batch['redacted'][i],
                    'crop': batch['crop'][i],
                })
        if all(len(v) >= n_per_class for v in collected.values()):
            break
    return collected


@torch.no_grad()
def deepfake_latent(unet, fruit_encoder, scheduler, vae, test_loader, device, n=6):
    unet.eval()
    fruit_encoder.eval()

    samples = find_samples_per_class(test_loader, n_per_class=n//2)
    n_each = min(n // 2, len(samples[0]), len(samples[1]))

    # Target backgrounds from apples, then oranges
    target_comps = torch.stack([s['composite'] for s in samples[0][:n_each]] +
                                [s['composite'] for s in samples[1][:n_each]]).to(device)
    target_masks = torch.stack([s['mask'] for s in samples[0][:n_each]] +
                                [s['mask'] for s in samples[1][:n_each]]).to(device)
    target_redacted = torch.stack([s['redacted'] for s in samples[0][:n_each]] +
                                  [s['redacted'] for s in samples[1][:n_each]]).to(device)
    target_labels = [0] * n_each + [1] * n_each

    # Source crops: oranges then apples (swapped)
    source_crops = torch.stack([s['crop'] for s in samples[1][:n_each]] +
                                [s['crop'] for s in samples[0][:n_each]]).to(device)
    source_labels = [1] * n_each + [0] * n_each

    # Generate in latent space
    z_result = scheduler.sample(unet, fruit_encoder, source_crops, target_redacted, target_masks)

    # Decode to pixel space
    orig_images = decode_latents(vae, target_comps)
    gen_images = decode_latents(vae, z_result)
    # Decode source crops for display
    source_images = decode_latents(vae, source_crops)
    redacted_images = decode_latents(vae, target_redacted)

    actual_n = len(target_labels)
    fig, axes = plt.subplots(4, actual_n, figsize=(3 * actual_n, 11))
    row_labels = ['Original', 'Redacted\n(background)', 'Source fruit\n(conditioning)', 'DEEPFAKE']

    for i in range(actual_n):
        axes[0, i].imshow(orig_images[i].cpu().permute(1, 2, 0).numpy())
        axes[0, i].set_title(CLASS_NAMES[target_labels[i]], fontsize=10, color='green')

        axes[1, i].imshow(redacted_images[i].cpu().permute(1, 2, 0).numpy())

        axes[2, i].imshow(source_images[i].cpu().permute(1, 2, 0).numpy())
        axes[2, i].set_title(CLASS_NAMES[source_labels[i]], fontsize=10, color='blue')

        axes[3, i].imshow(gen_images[i].cpu().permute(1, 2, 0).numpy())
        axes[3, i].set_title(
            f'{CLASS_NAMES[target_labels[i]]}→{CLASS_NAMES[source_labels[i]]}',
            fontsize=10, color='red'
        )

        for row in range(4):
            axes[row, i].axis('off')

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, labelpad=80, va='center')

    plt.suptitle('Latent Diffusion Deepfakes', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

deepfake_latent(unet, fruit_encoder, scheduler, vae, test_loader, device)

### Step 8 — Catching the Fakes 🕵️‍♂️
Now that we have successfully created a model capable of generating convincing fruit deepfakes, it's time to put on our detective hats! In the real world, as generative AI gets better at creating synthetic media, we also need robust tools to detect what is real and what is AI-generated.

To accomplish this, we're going to hand our U-Net one last job: generate a batch of deepfake fruits and package them up as a zip file so you can download them. 📦

Once you have the zip, switch over to the **discriminator notebook** linked in the MOODLE page — that's where we'll build and train our fruit lie-detector, mixing these fresh deepfakes with real fruits from the dataset. By the end of that notebook, you'll have taken your first real steps into AI safety and adversarial defense. Let's generate the evidence and see if our U-Net can fool its own judge! 🍎🍊🤔🔍

In [ ]:
#@title Generate deepfakes and download zip { display-mode: "form" }

import zipfile
from PIL import Image
from io import BytesIO

N_FAKES = 100
APPLE = 0
ORANGE = 1

@torch.no_grad()
def generate_deepfakes_for_discriminator(unet, fruit_encoder, scheduler, vae,
                                          loader, device, n_fakes=N_FAKES):
    """
    Generate n_fakes deepfake images by swapping fruit conditioning in latent
    space, then decoding through the frozen VAE back to pixel images (128x128).
    """
    unet.eval()
    fruit_encoder.eval()

    # Collect latent source crops per class for conditioning
    source_crops = {APPLE: [], ORANGE: []}
    for batch in loader:
        labels = batch['label']
        for cls_idx in [APPLE, ORANGE]:
            if len(source_crops[cls_idx]) >= 20:
                continue
            mask_cls = labels == cls_idx
            if mask_cls.any():
                cls_crops = batch['crop'][mask_cls]
                source_crops[cls_idx].extend(cls_crops)
        if all(len(v) >= 20 for v in source_crops.values()):
            break
    for k in source_crops:
        source_crops[k] = torch.stack(source_crops[k][:20])

    fake_pil_images = []
    generated = 0

    for batch in loader:
        if generated >= n_fakes:
            break

        z_redacted_batch = batch['redacted'].to(device)
        z_mask_batch = batch['mask'].to(device)
        labels = batch['label']

        for i in range(len(labels)):
            if generated >= n_fakes:
                break

            lbl = labels[i].item()
            opposite = ORANGE if lbl == APPLE else APPLE

            z_redacted = z_redacted_batch[i:i+1]
            z_mask = z_mask_batch[i:i+1]

            src_idx = random.randint(0, len(source_crops[opposite]) - 1)
            source_crop = source_crops[opposite][src_idx:src_idx+1].to(device)

            # Sample in latent space, then decode through the frozen VAE
            z_fake = scheduler.sample(unet, fruit_encoder, source_crop, z_redacted, z_mask)
            fake = decode_latents(vae, z_fake).clamp(0, 1)

            fake_np = (fake[0].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            fake_pil_images.append(Image.fromarray(fake_np))
            generated += 1

            if generated % 25 == 0:
                print(f"  Generated {generated}/{n_fakes} fakes...")

    return fake_pil_images


print(f"Generating {N_FAKES} deepfake images...")
fake_pil_images = generate_deepfakes_for_discriminator(
    unet, fruit_encoder, scheduler, vae, train_loader, device, N_FAKES
)

# Save as zip
zip_path = 'deepfake_imgs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for i, img in enumerate(fake_pil_images):
        buf = BytesIO()
        img.save(buf, format='PNG')
        zf.writestr(f'deepfake_imgs/fake_{i:04d}.png', buf.getvalue())

print(f"Saved {len(fake_pil_images)} images to {zip_path}")

# Download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("Download started!")
except ImportError:
    print(f"Not running in Colab — zip saved locally at {zip_path}")


In [ ]:
def save_checkpoint(unet, fruit_encoder, history, filename='diffusion_checkpoint.pt'):
    torch.save({
        'unet_state': unet.state_dict(),
        'encoder_state': fruit_encoder.state_dict(),
        'history': history,

    }, filename)
    print(f"Checkpoint saved to {filename}")

save_checkpoint(unet, fruit_encoder, history)